# Total cholesterol at or above the screening cut — Logistic Regression · PNS 2013

**FAPESP–Illinois undiagnosed NCD project** — Isabela Venancio da Silva (USP São Paulo) ·
Xuan Lin · Amogh Mannava (UIUC).

One model family, one outcome. Twelve notebooks share this shape, so any two of
them can be compared line for line.

| | |
|---|---|
| **Outcome** | `Z031` >= 200 mg/dL |
| **Threshold authority** | Conventional screening cut. SBC uses risk-stratified LDL targets rather than one diagnostic value |
| **Cohort** | adults who answer *no* to `Q060` (1 yes, 2 no), i.e. never told by a doctor they had this condition |
| **Expected build** | 5,927 rows, prevalence 32.1% |
| **Model** | Logistic Regression |
| **Dropped as condition-specific** | `dx_cholesterol` |

**Why this family.** The champion family in the 06/08/2026 run (AUC 0.739) and the model the paper is written around: coefficients read directly, and age enters as a natural cubic spline because risk is not linear in the log-odds.

## What this notebook does not decide

Preprocessing is frozen. Every variable decision comes from
`PNS_preprocessing_registry_v1.xlsx` through `pns_preprocess.py`: 117 declared
variables in, 91 out, 28 dropped, 2 composites, 5 renamed. **To change a
variable, change the workbook, not this notebook.**

Layer 1 (`build_matrix`) runs once and is deterministic. Layer 2
(`build_preprocessor`) — imputation, splines, encoding, scaling — is fitted
inside each cross-validation fold and never sees the test rows.

## Open with the group

PNS 2013 has no lipid-lowering medication item: `Q06204` records a recommendation, not use. This model cannot define 'treated' the way the other two can, so no treated or untreated subgroup is reported for this outcome. Recorded as a limitation in `05_article/Methods.docx`.

The lipid definition itself is still open in the decision log: TC, LDL, HDL or any abnormal lipid give between 360 and 2,331 positives.

`n_medications` and `n_chronic` are sums over their blocks and are rebuilt by
`build_matrix` **after** the condition-specific drop. This is the trap that once
produced an AUC of 0.86 with sensitivity 1.000; do not compute either counter
anywhere in this notebook.

## What the group decided, 22/09/2026

Four questions were open in the `notes` sheet. All four are now closed, and each
one is a change to the workbook rather than to any code in this notebook.

1. **`last_bp_measure` and `last_glucose_test` return**, symmetrically:
   `leak_hypertension` and `leak_diabetes` set to 0 for the respective item.
2. **Pregnancy-only diagnoses leave the base.** `Q002 = 2` and `Q030 = 2` make
   the gate undefined instead of counting as diagnosed, and `P005 = 3`, an
   undefined pregnancy, is excluded alongside `P005 = 1`.
3. **Insulin is declared** as `med_diabetes_insulin` (`Q03402`), condition-specific
   for diabetes only.
4. **Cholesterol has no treated definition**, recorded as a limitation in the
   Methods.

Three things also came out of testing the shared build and are fixed in
`pns_modelkit.py`, documented at the point of use: the diagnosis gates were being
imputed, which inflated two cohorts; the one-hot columns and their declared
categories disagreed on type, so Layer 2 could not fit; and three ordinal orders
are declared in the pre-reversal order that `_recode_fixes` already reversed.

`Q031`, `Q061` and `Q06201`–`Q06206` appear in the `outcomes` sheet as
condition-specific, but were never declared in the registry, so there is nothing
to drop.

---
# 1 · Setup

Installs, the data, and the run switches. Nothing here touches the science,
except section 1.3, where the research question is chosen.

## 1.1 · Install

**What this does.** Installs what Colab does not ship for this family: `optuna`,
plus `openpyxl` for the workbook.

**What to look for.** Nothing, unless it errors.

In [ ]:
!pip install -q optuna openpyxl

## 1.2 · Data and shared code

**What this does.** Clones the public repository, which holds the survey file,
both dictionaries, the frozen preprocessing (`pns_preprocess.py` and the
registry workbook) and the shared model helper (`pns_modelkit.py`).

**Drive is off by default**, since mounting asks for permission every session.
The run then writes to the session disk and zips itself at the end. Set
`MOUNT_DRIVE = True` to write into the shared folder instead, which also keeps
the built matrix between sessions and saves the minute it takes to rebuild.
Leave the `DRIVE = ...` line in place either way: the configuration cell reads
it whether or not anything is mounted.

**What to look for.** The two sha256 stamps printed by the build in section 2
must match across notebooks. They are what proves twelve runs used one matrix.

In [ ]:
import os, subprocess, sys

REPO_URL = "https://github.com/isasaade-23/pns2013-lab-exams-en.git"
REPO     = "/content/pns2013-lab-exams-en"

# Clone, or update a clone this session already has: a session that started
# before the last commit would otherwise keep running the old pns_modelkit.
if os.path.exists(REPO):
    subprocess.run(["git", "-C", REPO, "fetch", "-q", "--depth", "1", "origin", "main"])
    subprocess.run(["git", "-C", REPO, "reset", "-q", "--hard", "origin/main"])
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO], check=True)
sys.path.insert(0, os.path.join(REPO, "pipeline"))

print("pipeline at", subprocess.run(["git", "-C", REPO, "log", "-1", "--format=%h %s"],
                                    capture_output=True, text=True).stdout.strip())

DATA     = os.path.join(REPO, "data", "pns2013_lab_exams.xlsx")
REGISTRY = os.path.join(REPO, "pipeline", "PNS_preprocessing_registry_v1.xlsx")

# Drive is optional and off by default, because mounting asks for permission
# every session. Set MOUNT_DRIVE = True to write straight into the shared
# folder and to keep the built matrix between sessions; left False, the run
# writes to the session disk and zips itself at the end.
#
# DRIVE is defined either way. Do not comment this line out: the configuration
# cell below reads it.
MOUNT_DRIVE = False
DRIVE = "/content/drive/MyDrive/FAPESP_Illinois"

if MOUNT_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as e:
        print("Drive not mounted:", e)

print("data    ", os.path.exists(DATA))
print("registry", os.path.exists(REGISTRY))

## 1.3 · Configuration

**What this does.** Replaces what used to be command-line flags. Everything
downstream reads these.

| Switch | Meaning |
|---|---|
| `THRESHOLD` | `None` uses the guideline cut. The prespecified sensitivity analysis is `THRESHOLD = (190,)`; the LDL variant needs `Z033` and a registry change |
| `COHORT` | `"undiagnosed"` is the primary framing (Model A). `"all"` is diagnostic only |
| `N_TRIALS` | tuning budget. 40 finishes inside a free Colab session; `FULL = True` raises it to 150 |
| `REPEATS` | how many times the 5-fold split is repeated inside the search. 2 halves the noise in the objective and doubles the cost |
| `SCORING` | what the search ranks by. `roc_auc` for comparability; `average_precision` weighs the minority class, which is the honest target for diabetes at 3.8% |
| `RS` | 42, fixed, so the split is identical across the twelve notebooks |

**Runtime.** about 10 minutes at 40 trials with REPEATS = 2, 40 at 150.

In [ ]:
import importlib
import numpy as np, pandas as pd
import pns_preprocess as pp
import pns_modelkit as mk

# re-import, in case an older copy was imported earlier in this session
importlib.reload(pp); importlib.reload(mk)

OUTCOME   = "cholesterol"
MODEL     = "logreg"
THRESHOLD = None          # None = guideline default; see the table above
COHORT    = "undiagnosed"
FULL     = False          # True -> 150 trials, the paper run
N_TRIALS = 150 if FULL else 40
REPEATS  = 2              # 5-fold repeated this many times in the search
SCORING  = "roc_auc"      # "average_precision" ranks by the minority class

# outputs: the shared folder when Drive is mounted, the session disk otherwise
BASE   = f"{DRIVE}/02_analysis" if os.path.isdir(DRIVE) else "/content/work"
OUTDIR = os.path.join(BASE, "outputs", "models")
BUILD  = os.path.join(BASE, "outputs", "matrices")
os.makedirs(OUTDIR, exist_ok=True); os.makedirs(BUILD, exist_ok=True)

print("writing to", OUTDIR)

---
# 2 · The frozen matrix

**What this does.** Calls `build_matrix()` from `pns_preprocess.py` — or reuses
a cached build whose data and registry hashes still match — then asserts the
counts reported to the group: **5,927 rows at 32.1%**.

**What to look for.** If the assertion fails, stop. It means the registry or the
survey file moved, and no result from this notebook is comparable to the others
until that is understood.

In [ ]:
bundle = mk.load_or_build(OUTCOME, data=DATA, registry=REGISTRY,
                          outdir=BUILD, threshold=THRESHOLD, cohort=COHORT)
mk.check_frozen(bundle)

X, y = bundle["X"], bundle["y"]
print(f"\n{X.shape[0]} rows x {X.shape[1]} predictors, prevalence {y.mean():.1%}")
print("threshold authority:", bundle["threshold_authority"])
print("data sha", bundle["data_sha"], "| registry sha", bundle["registry_sha"])

a = mk.attrition(bundle, BUILD)
display(a) if a is not None else None

**Participant flow and roles.** The attrition table above is the row accounting
for the flow diagram. Below, where each column enters Layer 2.

In [ ]:
for k, v in bundle["roles"].items():
    print(f"{k:<10} {len(v):>3}  {', '.join(v[:6])}{' ...' if len(v) > 6 else ''}")

blocks = pd.Series({c: bundle["spec"][c]["block"]
                    for c in X.columns if c in bundle["spec"]}).value_counts()
print("\npredictors per block\n", blocks.to_string())

---
# 3 · Split and preprocessor

**What this does.** Stratified 80/20 at `random_state=42`, then builds the Layer 2
`ColumnTransformer`. For Logistic Regression: scaling **on**,
age spline **on**.

**What to look for.** Train and test prevalence should match to a decimal. The
preprocessor is passed *into* the pipeline, never fitted here.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import (RepeatedStratifiedKFold, StratifiedKFold,
                                     cross_val_score)

Xtr, Xte, ytr, yte = mk.split(bundle)
SPW = mk.pos_weight(ytr)

# REPEATS > 1 gives the search a quieter target to optimise. One 5-fold pass is
# noisy enough that a lucky split can outrank a better model.
cv = (RepeatedStratifiedKFold(n_splits=5, n_repeats=REPEATS, random_state=mk.RS)
      if REPEATS > 1 else
      StratifiedKFold(n_splits=5, shuffle=True, random_state=mk.RS))
cv_report = StratifiedKFold(n_splits=5, shuffle=True, random_state=mk.RS)

def preproc():
    return mk.preprocessor(bundle, spline=True, scale=True)

n_encoded = mk.preprocessor(bundle, spline=True, scale=True,
                            verbose=True).fit_transform(Xtr).shape[1]
print(f"{X.shape[1]} raw predictors -> {n_encoded} encoded columns")
print(f"positives: train {int(ytr.sum())}, test {int(yte.sum())} "
      f"(scale_pos_weight {SPW:.1f})")

---
# 4 · Fit

Two fits, reported side by side: library defaults, and an Optuna TPE search over
5-fold AUC. The comparison is the honest way to say whether tuning bought
anything — in the hypertension run it was worth about +0.006 AUC.

## 4.1 · Defaults

**What to look for.** The CV AUC here is the floor. For hypertension it should
land near 0.730; the tuned run near 0.736.

In [ ]:
from sklearn.linear_model import LogisticRegression

def make_est(params):
    return LogisticRegression(max_iter=4000, solver='liblinear',
                              random_state=mk.RS, **params)

pipe_def = Pipeline([("prep", preproc()),
                     ("clf", LogisticRegression(max_iter=3000, class_weight='balanced',
                            solver='liblinear', random_state=mk.RS))])

cv_def = cross_val_score(pipe_def, Xtr, ytr, cv=cv_report,
                         scoring=SCORING, n_jobs=1)
print(f"default 5-fold CV AUC {cv_def.mean():.3f} +/- {cv_def.std():.3f}")

pipe_def.fit(Xtr, ytr)
row_def, p_def = mk.evaluate(pipe_def, Xte, yte, "Logistic Regression (default)")
print(f"test AUC {row_def['AUC_test']:.3f} "
      f"[{row_def['AUC_lo']:.3f}, {row_def['AUC_hi']:.3f}]")

## 4.2 · Tuned

**The slow cell.** About 10 minutes at 40 trials with repeats = 2, 40 at 150. The objective reports fold by
fold so the pruner can stop a hopeless trial early. The search space is the one
used in the 07/08/2026 revision, unchanged.

In [ ]:
import optuna
from sklearn.metrics import roc_auc_score
optuna.logging.set_verbosity(optuna.logging.WARNING)

def suggest(t):
    # elasticnet was tried and dropped: it needs the saga solver, which
    # takes minutes per fit on 162 encoded columns, and l1 already won
    # the August run at C around 0.015
    return {'C': t.suggest_float('C', 1e-4, 1e3, log=True),
            'penalty': t.suggest_categorical('penalty', ['l1', 'l2']),
            # weighting the minority class is a hyper-parameter, not a
            # setting: 'balanced' helps recall and can cost ranking
            'class_weight': t.suggest_categorical('class_weight',
                                                 ['balanced', None])}

from sklearn.metrics import average_precision_score
SCORE = {"roc_auc": roc_auc_score, "average_precision": average_precision_score}[SCORING]

def objective(trial):
    """Mean score over REPEATS x 5 folds.

    One 5-fold pass is a noisy target to optimise, and the search will happily
    chase that noise: at 3.8% prevalence a validation fold holds about fifty
    positives, so a fold-to-fold swing of 0.03 in AUC is ordinary. Repeating the
    split with different shuffles averages the swing down, which is the
    difference between selecting a better model and selecting a luckier split.
    """
    params, scores = suggest(trial), []
    for k, (itr, iva) in enumerate(cv.split(Xtr, ytr)):
        pipe = Pipeline([("prep", preproc()), ("clf", make_est(params))])
        pipe.fit(Xtr.iloc[itr], ytr.iloc[itr])
        scores.append(SCORE(ytr.iloc[iva], pipe.predict_proba(Xtr.iloc[iva])[:, 1]))
        trial.report(float(np.mean(scores)), k)
        if trial.should_prune():
            raise optuna.TrialPruned()
    return float(np.mean(scores))

study = optuna.create_study(direction="maximize",
                            sampler=optuna.samplers.TPESampler(seed=mk.RS,
                                                               multivariate=True),
                            pruner=optuna.pruners.MedianPruner(n_warmup_steps=5))
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

best = study.best_params
done = [t for t in study.trials if t.value is not None]
spread = np.std([t.value for t in done])
print(f"\nbest CV {SCORING} {study.best_value:.3f} after {len(study.trials)} trials "
      f"({len(done)} completed, {len(study.trials) - len(done)} pruned)")
print(f"spread across completed trials: {spread:.3f}")
print(best)

pipe_tuned = Pipeline([("prep", preproc()),
                       ("clf", make_est(suggest(optuna.trial.FixedTrial(best))))])
pipe_tuned.fit(Xtr, ytr)
row_tuned, p_tuned = mk.evaluate(pipe_tuned, Xte, yte, "Logistic Regression (tuned)")
row_def["CV_AUC"], row_tuned["CV_AUC"] = cv_def.mean(), study.best_value

# The gap between what the search selected on and what the test half gives.
# A search that gained on cross-validation and lost on test was fitting the
# folds, and the honest reading is that tuning bought nothing.
print(f"\ntuning moved CV by {study.best_value - cv_def.mean():+.3f} "
      f"and test by {row_tuned['AUC_test'] - row_def['AUC_test']:+.3f}")
print(f"CV minus test for the tuned model: "
      f"{study.best_value - row_tuned['AUC_test']:+.3f}")
BEST_MODEL, BEST_P, BEST_PARAMS = pipe_tuned, p_tuned, best

---
# 5 · Four variants of the same model

The tuned model is refitted three more ways, to answer whether the class
imbalance is what is holding the numbers down.

| Variant | What changes |
|---|---|
| `base` | the tuned model, unchanged |
| `smote` | `SMOTE` inside the pipeline, after the preprocessor, fitted only on training rows |
| `bagging` | `BalancedBaggingClassifier`: each member sees a bootstrap resample with the classes balanced |
| `calib` | isotonic recalibration of `base` by internal cross-validation |

**The search runs once, not four times.** The hyper-parameters come from the
`base` study and the variants change the sampling, not the model. Four searches
would cost four times as much and would confound the difference between variants
with the variation of the search itself.

**What to expect, so the result can contradict it.** Resampling does not create
information, so AUC rarely moves: SMOTE and bagging invent or duplicate cases,
they do not add signal. What they do change is the probability scale. A model
trained on a resampled world where half the people have the outcome reports the
risk of a population that does not exist, and the calibration intercept moves
away from zero. This is why the metrics table carries `calib_intercept` and
`calib_slope` alongside AUC: AUC is invariant to any monotone rescaling of the
probabilities and will not notice calibration breaking.

If the goal is a better-calibrated risk, `calib` is the variant built for it.
If the goal is discrimination, the honest prior is that none of the three moves
it, and the lever is elsewhere — in the outcome definition, not the sampler.

In [ ]:
!pip install -q imbalanced-learn

In [ ]:
from imblearn.over_sampling import SMOTE
from imblearn.ensemble import BalancedBaggingClassifier
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.calibration import CalibratedClassifierCV

def _final_estimator():
    """A fresh copy of the estimator the search selected."""
    return make_est(suggest(optuna.trial.FixedTrial(best)))

VARIANTS = {}

def register(name, model_obj):
    """Fit, evaluate, and keep the probabilities for the figures."""
    model_obj.fit(Xtr, ytr)
    row, prob = mk.evaluate(model_obj, Xte, yte, f"{MODEL} ({name})")
    row["variant"] = name
    VARIANTS[name] = dict(model=model_obj, prob=prob, row=row)
    print(f"{name:<8} AUC {row['AUC_test']:.3f} "
          f"[{row['AUC_lo']:.3f}, {row['AUC_hi']:.3f}]  "
          f"Brier {row['Brier']:.4f}  "
          f"calibration intercept {row['calib_intercept']:+.2f} "
          f"slope {row['calib_slope']:.2f}")
    return row

# base: the model section 4 already fitted
row_base = row_tuned.copy()
row_base["variant"] = "base"
VARIANTS["base"] = dict(model=BEST_MODEL, prob=BEST_P, row=row_base)
print(f"{'base':<8} AUC {row_base['AUC_test']:.3f} "
      f"[{row_base['AUC_lo']:.3f}, {row_base['AUC_hi']:.3f}]  "
      f"Brier {row_base['Brier']:.4f}  "
      f"calibration intercept {row_base['calib_intercept']:+.2f} "
      f"slope {row_base['calib_slope']:.2f}")

# smote: resampling belongs after the encoder. Synthesising a case between two
# rows of raw questionnaire codes would invent categories that do not exist.
#
# The preprocessor's steps are spread into the chain rather than nested, because
# imblearn refuses a Pipeline as an intermediate step.
register("smote", ImbPipeline([*preproc().steps,
                               ("smote", SMOTE(random_state=mk.RS, k_neighbors=5)),
                               ("clf", _final_estimator())]))

if True :
    # The preprocessor stays outside the ensemble: imblearn refuses a Pipeline
    # as a bagged estimator, and resampling belongs on encoded columns anyway,
    # exactly where SMOTE sits.
    register("bagging", Pipeline([
        ("prep", preproc()),
        ("clf", BalancedBaggingClassifier(
            estimator=_final_estimator(), n_estimators=10,
            sampling_strategy="auto", replacement=False,
            random_state=mk.RS, n_jobs=1))]))

    register("calib", CalibratedClassifierCV(
        Pipeline([("prep", preproc()), ("clf", _final_estimator())]),
        method="isotonic", cv=5))

---
# 6 · Results

Measured on the test half, which nothing above was fitted on.

## 6.1 · Metrics

**What to look for.** `AUC_lo`/`AUC_hi` is a 1,000-draw stratified bootstrap
interval on the test AUC. Two operating points are reported: Youden's J, and the
screening point that holds sensitivity at about 90% — the one a screening
instrument would actually be set at, and the one where PPV shows what the
prevalence costs.

`calib_intercept` and `calib_slope` are where the variants separate. Zero and
one is perfect. An intercept below zero means the model reports a risk higher
than the one observed, which is what training on resampled data produces.

In [ ]:
row_def["variant"] = "untuned"
results = mk.metrics_frame([row_def] + [v["row"] for v in VARIANTS.values()])
display(results)

print("\ncalibration, one line per variant")
cal = pd.DataFrame([{"variant": k,
                     "AUC": v["row"]["AUC_test"],
                     "Brier": v["row"]["Brier"],
                     "intercept": v["row"]["calib_intercept"],
                     "slope": v["row"]["calib_slope"],
                     "mean predicted": v["row"]["mean_predicted"],
                     "observed": v["row"]["observed"]}
                    for k, v in VARIANTS.items()])
display(cal.round(4))

## 6.2 · Confusion matrices

The 2x2 at both operating points, for every variant. The percentage in each cell
is the share of its **true** row: of the people who have the outcome, how many
the model flags, and of the people who do not, how many it flags anyway.

**What to look for.** At Youden the matrix is balanced by construction and says
little on its own. The screening point is the one that matters: holding
sensitivity near 90% in a cohort where the outcome is uncommon means flagging a
large share of everyone, and the false-positive cell is the cost of the
instrument.

In [ ]:
for name, v in VARIANTS.items():
    for point, label in ((v["row"]["youden_threshold"], "Youden"),
                         (v["row"]["screen90_threshold"], "~90% sensitivity")):
        print(f"\n{name} · {label}")
        display(mk.confusion_frame(yte, v["prob"], point))

figs_cm = {}
for name, v in VARIANTS.items():
    figs_cm[name] = mk.plot_confusion(
        yte, v["prob"], v["row"]["screen90_threshold"],
        f"{OUTCOME} · {MODEL} · {name} · screening point")

## 6.3 · Permutation importance

**What this does.** Shuffles one predictor at a time in the test set and measures
how far AUC falls, then sums the drops per registry block. Ten repeats over every predictor.

**What to look for.** The block table is the null-result table of the decision
log: for hypertension, access to care contributed exactly 0.000 and sleep less
than that. A block that suddenly matters for a new outcome is a finding; a block
that matters *too much* is usually leakage, and the first thing to check is
whether a counter was rebuilt.

In [ ]:
imp, blocks_imp = mk.permutation_report(BEST_MODEL, Xte, yte, bundle, n_repeats=10)
display(imp.head(20))
display(blocks_imp)

## 6.4 · Figures

One ROC with all four variants, the calibration curve of `base`, and the top 20
predictors. Written at 300 dpi.

Four curves that lie on top of each other is the expected picture, and it is the
answer to the question that started this: resampling moves the probabilities,
not the ranking.

In [ ]:
fig_roc = mk.plot_roc({name: (yte, v["prob"]) for name, v in VARIANTS.items()},
                      f"{OUTCOME} · {MODEL} · four variants")
fig_cal = mk.plot_calibration(yte, BEST_P, f"{OUTCOME} · calibration, base")
fig_imp = mk.plot_importance(imp, f"{OUTCOME} · permutation importance")

---
# 7 · Export

One folder per variant, under `outputs/models/<outcome>/<family>_<variant>/`:
the metrics row, the importance tables, the best parameters, the figures and a
plain-text report carrying the build hashes.

**The zip carries a timestamp**, `<outcome>_<family>_<variant>_<YYYYMMDD-HHMM>.zip`,
so two runs of the same cell do not overwrite each other in the Downloads folder
and the comparison notebook can keep the most recent of each combination.

**Where it goes.** `PUSH_RESULTS = True` commits the folder to
[isasaade-23/pns2013-model-runs](https://github.com/isasaade-23/pns2013-model-runs),
which is private — these are unpublished results. It needs a GitHub token in
the Colab saved keys named `GITHUB_TOKEN`: a fine-grained token with
**Contents: read and write** on that repository, enabled for this notebook.
Without a token, or with `PUSH_RESULTS = False`, the run zips itself into
`outputs/models/` instead, next to the folder it just wrote. That zip only goes
through the browser when there is nowhere durable to keep it: with
`MOUNT_DRIVE = True` it lands in the shared folder and nothing is downloaded,
and without Drive it lives on the session disk, which dies with the session, so
a copy is downloaded as well.

In [ ]:
import datetime

PUSH_RESULTS = False       # True -> commit to the private run repository as well
STAMP = datetime.datetime.utcnow().strftime("%Y%m%d-%H%M")

for name, v in VARIANTS.items():
    tag = f"{MODEL}_{name}"
    is_base = name == "base"
    folder = mk.export(
        OUTDIR, OUTCOME, tag, bundle,
        results[results["variant"] == name],
        importance=imp if is_base else None,
        blocks=blocks_imp if is_base else None,
        best_params={**BEST_PARAMS, "variant": name},
        figures=([("roc", fig_roc), ("calibration", fig_cal),
                  ("importance", fig_imp)] if is_base else [])
                + [("confusion", figs_cm[name])])

    if PUSH_RESULTS:
        try:
            mk.push_results(folder)
            continue
        except Exception as e:
            print("not pushed:", e)

    z = mk.zip_folder(folder, os.path.join(OUTDIR, f"{OUTCOME}_{tag}_{STAMP}.zip"))
    if not os.path.isdir(DRIVE):
        try:
            from google.colab import files
            files.download(z)
        except Exception as e:
            print(e)

results.to_csv(os.path.join(OUTDIR, f"table_{OUTCOME}_{MODEL}_{STAMP}.csv"),
               index=False)
print(f"\n{len(VARIANTS)} variants exported, stamp {STAMP}")

## 7.1 · Re-push a run that is already on disk

**What this does.** Sends the folder the cell above wrote, without refitting
anything. Use it when the export did not push: a session that cloned before the
last commit and ran an older `pns_modelkit`, a missing token, a connection that
dropped. Set `RETRY_PUSH = True` and run this cell alone.

**What to look for.** The link it prints. If it says no token, add `GITHUB_TOKEN`
to the Colab saved keys and run it again — the results are on disk either way,
and nothing has to be recomputed.

In [ ]:
RETRY_PUSH = False

if RETRY_PUSH:
    import importlib
    subprocess.run(["git", "-C", REPO, "fetch", "-q", "--depth", "1", "origin", "main"])
    subprocess.run(["git", "-C", REPO, "reset", "-q", "--hard", "origin/main"])
    import pns_modelkit as mk
    importlib.reload(mk)
    for name in VARIANTS:
        mk.push_results(os.path.join(OUTDIR, OUTCOME, f"{MODEL}_{name}"))